In [6]:
import json
import sys
from base64 import b64encode
from datetime import datetime
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

cwd = Path.cwd().resolve()
if (cwd / "common.py").exists():
    AI_DIR = cwd
elif (cwd.parent / "common.py").exists():
    AI_DIR = cwd.parent
elif (cwd / "AI" / "common.py").exists():
    AI_DIR = cwd / "AI"
else:
    raise FileNotFoundError("Cannot locate AI/common.py")

if str(AI_DIR) not in sys.path:
    sys.path.insert(0, str(AI_DIR))

from common import load_twilio_config


In [7]:
config = load_twilio_config()

def mask_number(number: str) -> str:
    if len(number) <= 6:
        return number
    return f"{number[:4]}...{number[-3:]}"

TEST_TO_NUMBER = config["TWILIO_TO_NUMBER"]
TEST_FROM_NUMBER = config["TWILIO_FROM_NUMBER"]
TEST_MESSAGE = f"[Notebook SMS Test] {datetime.now().isoformat(timespec='seconds')}"

print("Account SID:", config["TWILIO_ACCOUNT_SID"])
print("From:", mask_number(TEST_FROM_NUMBER))
print("To:", mask_number(TEST_TO_NUMBER))
print("Message:", TEST_MESSAGE)


Account SID: ACdcd553852afef1a5d6890c2e0432fac8
From: +158...807
To: +849...025
Message: [Notebook SMS Test] 2026-03-21T15:07:32


In [8]:
def send_twilio_sms(account_sid: str, auth_token: str, to_number: str, from_number: str, body: str):
    url = f"https://api.twilio.com/2010-04-01/Accounts/{account_sid}/Messages.json"
    payload = urlencode({
        "To": to_number,
        "From": from_number,
        "Body": body,
    }).encode("utf-8")

    auth = b64encode(f"{account_sid}:{auth_token}".encode("utf-8")).decode("ascii")
    request = Request(
        url,
        data=payload,
        headers={
            "Authorization": f"Basic {auth}",
            "Content-Type": "application/x-www-form-urlencoded",
            "Accept": "application/json",
        },
        method="POST",
    )

    try:
        with urlopen(request, timeout=20) as response:
            raw_text = response.read().decode("utf-8")
            return {
                "ok": 200 <= response.status < 300,
                "http_status": response.status,
                "data": json.loads(raw_text),
                "raw_text": raw_text,
            }
    except HTTPError as exc:
        raw_text = exc.read().decode("utf-8")
        return {
            "ok": False,
            "http_status": exc.code,
            "data": json.loads(raw_text) if raw_text else {},
            "raw_text": raw_text,
        }
    except URLError as exc:
        return {
            "ok": False,
            "http_status": None,
            "data": {"message": str(exc.reason)},
            "raw_text": str(exc),
        }


In [9]:
result = send_twilio_sms(
    account_sid=config["TWILIO_ACCOUNT_SID"],
    auth_token=config["TWILIO_AUTH_TOKEN"],
    to_number=TEST_TO_NUMBER,
    from_number=TEST_FROM_NUMBER,
    body=TEST_MESSAGE,
)

print("HTTP status:", result["http_status"])
print(json.dumps(result["data"], indent=2, ensure_ascii=False))


HTTP status: 400
{
  "code": 21608,
  "message": "The number +8493140XXXX is unverified. Trial accounts cannot send messages to unverified numbers; verify +8493140XXXX at twilio.com/user/account/phone-numbers/verified, or purchase a Twilio number to send messages to unverified numbers",
  "more_info": "https://www.twilio.com/docs/errors/21608",
  "status": 400
}


In [10]:
data = result["data"]

if result["ok"] and data.get("sid"):
    print("Twilio accepted the request.")
    print("Message SID:", data.get("sid"))
    print("Initial status:", data.get("status"))
else:
    print("Twilio did not accept the request.")
    print("Error code:", data.get("code") or data.get("error_code"))
    print("Error message:", data.get("message"))


Twilio did not accept the request.
Error code: 21608
Error message: The number +8493140XXXX is unverified. Trial accounts cannot send messages to unverified numbers; verify +8493140XXXX at twilio.com/user/account/phone-numbers/verified, or purchase a Twilio number to send messages to unverified numbers
